# Synthea → FHIR → RAG: Clinical Decision Support Pipeline

This notebook demonstrates an end-to-end pipeline for building a **Retrieval-Augmented Generation (RAG)** clinical decision support system using synthetic patient data.

## Learning Objectives

By the end of this notebook, you will be able to:
1. Generate synthetic patient records using **Synthea**
2. Load FHIR bundles into a local **FHIR** server via REST API
3. Query FHIR resources and build structured patient summaries
4. Construct a RAG knowledge base from clinical guidelines using **ChromaDB**
5. Use a local LLM (**Ollama**) to generate guideline-grounded clinical reasoning
6. Compare RAG vs non-RAG responses to understand retrieval-augmented generation

## Architecture Overview

```mermaid
flowchart LR
    subgraph Data Generation
        A[Synthea] -->|FHIR JSON bundles| B[HAPI FHIR\nDocker]
    end

    subgraph Patient Data
        B -->|REST API\nGET/POST| C[Python\nrequests]
        C -->|Patient\nsummaries| D{RAG\nPipeline}
    end

    subgraph Knowledge Base
        E[Clinical\nGuidelines] -->|sentence-transformers\nembed| F[(numpy\nvectors)]
        F -->|cosine\nsimilarity| D
    end

    D -->|prompt +\ncontext| G[Ollama\nllama3.2\nLocal LLM]
    G -->|Clinical\nassessment| H[Output]

    style A fill:#4a9eff,color:#fff
    style B fill:#ff6b6b,color:#fff
    style F fill:#51cf66,color:#fff
    style G fill:#ffd43b,color:#000
```

**Key components:**
- **Synthea**: Open-source synthetic patient generator (FHIR R4 bundles)
- **HAPI FHIR**: Open-source FHIR server running in Docker
- **numpy + sentence-transformers**: Lightweight vector embeddings and cosine similarity search
- **Ollama**: Local LLM runtime (no API keys, no cloud, runs on any laptop)

## Background: RAG vs Fine-Tuning

There are two main approaches to adapting LLMs for domain-specific tasks:

| Aspect | RAG | Fine-Tuning |
|--------|-----|-------------|
| **How it works** | Retrieve relevant documents at query time, inject into prompt | Update model weights on domain data |
| **Knowledge updates** | Swap out documents instantly | Requires retraining |
| **Hallucination control** | Grounds answers in retrieved text | Model may still hallucinate |
| **Compute cost** | Low (embedding + inference) | High (GPU training) |
| **Best for** | Factual Q&A, guideline lookup | Style adaptation, classification |

This notebook uses **RAG** because clinical guidelines change frequently and we want answers grounded in specific, citable sources. See `FineTuning_GenAI.ipynb` for the fine-tuning approach.

---
## Step 1: Setup & Prerequisites

### External Requirements

Before running this notebook, ensure the following are running:

1. **Local Azure FHIR server** on `https://localhost:44348`

2. **Ollama with a model pulled:**
   ```bash
   ollama serve          # Start Ollama (if not already running)
   ollama pull llama3.2  # Download model (~2GB)
   ```

3. **Synthea** installed locally (download from https://github.com/synthetichealth/synthea)

In [1]:
# Install required packages (uncomment and run once)
# !pip install requests pandas langchain-ollama sentence-transformers

In [3]:
import os
import json
import glob
import subprocess
import platform
from pathlib import Path
from collections import Counter

import numpy as np
import requests
import pandas as pd
from langchain_ollama import OllamaLLM
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from sentence_transformers import SentenceTransformer

print("All imports successful.")

All imports successful.


In [ ]:
# ============================================================
# CONFIGURATION - Edit these paths for your environment
# ============================================================

# Path to your local Synthea installation (the directory containing run_synthea or gradlew)
SYNTHEA_PATH = "C:\\repos\\synthea"  # Windows example; use "/home/user/synthea" on Linux/Mac

# FHIR server base URL (local Azure FHIR server)
FHIR_BASE_URL = "https://localhost:44348"

# Ollama settings
OLLAMA_BASE_URL = "http://localhost:11434"
OLLAMA_MODEL = "llama3.2"  # ~2GB download; alternatives: mistral, phi3, gemma2

# Synthea generation settings
SYNTHEA_POPULATION = 10  # Number of patients to generate
SYNTHEA_STATE = "Washington"
SYNTHEA_CITY = "Seattle"

print("Configuration loaded.")
print(f"  Synthea path:  {SYNTHEA_PATH}")
print(f"  FHIR server:   {FHIR_BASE_URL}")
print(f"  Ollama model:  {OLLAMA_MODEL}")

In [ ]:
# Prerequisite checker
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Disable SSL verification for local dev FHIR server with self-signed certificate
VERIFY_SSL = False

def check_prerequisites():
    """Verify that all external services are reachable."""
    checks = {}

    # Check FHIR server
    try:
        r = requests.get(f"{FHIR_BASE_URL}/metadata", timeout=5, verify=VERIFY_SSL)
        checks["FHIR Server"] = r.status_code == 200
    except requests.ConnectionError:
        checks["FHIR Server"] = False

    # Check Ollama
    try:
        r = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=5)
        if r.status_code == 200:
            models = [m["name"] for m in r.json().get("models", [])]
            # Match model name with or without :latest tag
            has_model = any(m == OLLAMA_MODEL or m.startswith(f"{OLLAMA_MODEL}:") for m in models)
            checks["Ollama"] = True
            checks[f"Model '{OLLAMA_MODEL}'"] = has_model
            if not has_model:
                print(f"  Available models: {models}")
        else:
            checks["Ollama"] = False
    except requests.ConnectionError:
        checks["Ollama"] = False

    # Check Synthea path
    synthea_dir = Path(SYNTHEA_PATH)
    checks["Synthea directory"] = synthea_dir.exists()

    # Report
    all_ok = True
    for name, ok in checks.items():
        status = "OK" if ok else "MISSING"
        print(f"  [{status:>7s}] {name}")
        if not ok:
            all_ok = False

    if all_ok:
        print("\nAll prerequisites met.")
    else:
        print("\nSome prerequisites are missing. See setup instructions above.")

    return all_ok

check_prerequisites()

---
## Step 2: Generate Synthetic Patient Data with Synthea

[Synthea](https://github.com/synthetichealth/synthea) generates realistic (but fictional) patient records in FHIR R4 format. Each patient gets a complete medical history: conditions, medications, observations, encounters, procedures, and more.

In [ ]:
# Generate synthetic patients with Synthea
synthea_dir = Path(SYNTHEA_PATH)
output_dir = synthea_dir / "output" / "fhir"

# Build the Synthea command (cross-platform)
if platform.system() == "Windows":
    cmd = [str(synthea_dir / "run_synthea.bat")]
else:
    cmd = [str(synthea_dir / "run_synthea")]

cmd.extend([
    "-p", str(SYNTHEA_POPULATION),
    "--exporter.fhir.export", "true",
    "--exporter.hospital.fhir.export", "false",
    "--exporter.practitioner.fhir.export", "false",
    "--exporter.ccda.export", "false",
    SYNTHEA_STATE, SYNTHEA_CITY
])

print(f"Running Synthea to generate {SYNTHEA_POPULATION} patients...")
print(f"Command: {' '.join(cmd)}\n")

result = subprocess.run(cmd, cwd=str(synthea_dir), capture_output=True, text=True, timeout=300)

if result.returncode == 0:
    print("Synthea completed successfully.")
else:
    print(f"Synthea exited with code {result.returncode}")
    if result.stderr:
        print(f"Errors: {result.stderr[:500]}")

In [ ]:
# Explore Synthea output
bundle_files = sorted(glob.glob(str(output_dir / "*.json")))
print(f"Found {len(bundle_files)} FHIR bundle files in {output_dir}\n")

# Count resource types across all bundles
resource_counts = Counter()
for f in bundle_files:
    with open(f) as fh:
        bundle = json.load(fh)
        for entry in bundle.get("entry", []):
            rtype = entry.get("resource", {}).get("resourceType", "Unknown")
            resource_counts[rtype] += 1

print("Resource types across all bundles:")
for rtype, count in resource_counts.most_common():
    print(f"  {rtype:30s} {count:5d}")

In [ ]:
# Examine one bundle's structure
if bundle_files:
    with open(bundle_files[0]) as fh:
        sample_bundle = json.load(fh)

    print(f"Bundle type: {sample_bundle.get('type')}")
    print(f"Total entries: {len(sample_bundle.get('entry', []))}")
    print(f"\nFirst 10 resource types in this bundle:")

    for entry in sample_bundle["entry"][:10]:
        resource = entry["resource"]
        rtype = resource["resourceType"]
        # Show a summary field depending on type
        if rtype == "Patient":
            name = resource.get("name", [{}])[0]
            label = f"{name.get('given', [''])[0]} {name.get('family', '')}"
        elif rtype == "Condition":
            label = resource.get("code", {}).get("text", "N/A")
        elif rtype == "MedicationRequest":
            label = resource.get("medicationCodeableConcept", {}).get("text", "N/A")
        elif rtype == "Observation":
            label = resource.get("code", {}).get("text", "N/A")
        else:
            label = resource.get("id", "")[:40]
        print(f"  {rtype:25s} | {label}")
else:
    print("No bundle files found. Run Synthea first (cell above).")

---
## Step 3: Load Bundles into FHIR Server

Synthea generates **transaction bundles** — each bundle is a single patient's complete record. We POST each bundle to the FHIR server, which processes all resources in a single transaction.

In [ ]:
# POST each transaction bundle to FHIR server
load_results = []

for i, filepath in enumerate(bundle_files, 1):
    filename = os.path.basename(filepath)
    with open(filepath) as fh:
        bundle = json.load(fh)

    resp = requests.post(
        FHIR_BASE_URL,
        json=bundle,
        headers={"Content-Type": "application/fhir+json"},
        verify=VERIFY_SSL,
    )

    status = "OK" if resp.status_code in (200, 201) else f"FAIL ({resp.status_code})"
    entry_count = len(bundle.get("entry", []))
    load_results.append({"file": filename, "entries": entry_count, "status": status})
    print(f"  [{i}/{len(bundle_files)}] {filename}: {entry_count} resources -> {status}")

df_load = pd.DataFrame(load_results)
success_count = (df_load["status"] == "OK").sum()
print(f"\nLoaded {success_count}/{len(bundle_files)} bundles successfully.")

In [ ]:
# Verify: query patient count from FHIR server
resp = requests.get(f"{FHIR_BASE_URL}/Patient?_summary=count", verify=VERIFY_SSL)
total = resp.json().get("total", "unknown")
print(f"Patients on FHIR server: {total}")

---
## Step 4: Query FHIR & Build Patient Summaries

Now we query the FHIR server using raw REST calls. This is intentionally transparent — you see every HTTP request and JSON response, unlike higher-level FHIR client libraries.

We build **text summaries** of each patient's record, which become the input to our RAG system.

In [ ]:
def fhir_get(resource_type, params=None):
    """Query FHIR server with automatic pagination."""
    url = f"{FHIR_BASE_URL}/{resource_type}"
    all_entries = []

    while url:
        resp = requests.get(url, params=params, verify=VERIFY_SSL)
        resp.raise_for_status()
        bundle = resp.json()
        all_entries.extend(bundle.get("entry", []))

        # Follow pagination links
        params = None  # Only use params on first request
        url = None
        for link in bundle.get("link", []):
            if link["relation"] == "next":
                url = link["url"]
                break

    return [e["resource"] for e in all_entries]


def extract_patient_info(patient):
    """Extract key demographics from a Patient resource."""
    name = patient.get("name", [{}])[0]
    given = " ".join(name.get("given", []))
    family = name.get("family", "")
    return {
        "id": patient["id"],
        "name": f"{given} {family}".strip(),
        "gender": patient.get("gender", "unknown"),
        "birthDate": patient.get("birthDate", "unknown"),
    }


def extract_conditions(patient_id):
    """Get active conditions for a patient."""
    resources = fhir_get("Condition", {"patient": patient_id})
    conditions = []
    for r in resources:
        code_text = r.get("code", {}).get("text", "")
        status = r.get("clinicalStatus", {}).get("coding", [{}])[0].get("code", "")
        onset = r.get("onsetDateTime", r.get("recordedDate", "unknown"))
        if code_text:
            conditions.append({"condition": code_text, "status": status, "onset": onset[:10]})
    return conditions


def extract_medications(patient_id):
    """Get medication requests for a patient."""
    resources = fhir_get("MedicationRequest", {"patient": patient_id})
    meds = []
    for r in resources:
        med_text = r.get("medicationCodeableConcept", {}).get("text", "")
        status = r.get("status", "unknown")
        if med_text:
            meds.append({"medication": med_text, "status": status})
    return meds


def extract_observations(patient_id, max_per_type=3):
    """Get recent observations (labs/vitals) for a patient."""
    resources = fhir_get("Observation", {"patient": patient_id, "_sort": "-date", "_count": "100"})
    obs_by_type = {}
    for r in resources:
        code_text = r.get("code", {}).get("text", "")
        if not code_text:
            continue
        if code_text not in obs_by_type:
            obs_by_type[code_text] = []
        if len(obs_by_type[code_text]) < max_per_type:
            value = r.get("valueQuantity", {})
            val_str = f"{value.get('value', 'N/A')} {value.get('unit', '')}".strip()
            date = r.get("effectiveDateTime", "unknown")[:10]
            obs_by_type[code_text].append({"value": val_str, "date": date})
    return obs_by_type

print("FHIR helper functions defined.")

In [ ]:
# Query all patients from FHIR server
patients_raw = fhir_get("Patient")
patients = [extract_patient_info(p) for p in patients_raw]

df_patients = pd.DataFrame(patients)
print(f"Retrieved {len(patients)} patients:\n")
df_patients

In [ ]:
def build_patient_summary(patient_info):
    """Build a text summary of a patient's clinical record from FHIR data."""
    pid = patient_info["id"]
    lines = []
    lines.append(f"Patient: {patient_info['name']}")
    lines.append(f"Gender: {patient_info['gender']}, DOB: {patient_info['birthDate']}")

    # Conditions
    conditions = extract_conditions(pid)
    if conditions:
        lines.append("\nActive Conditions:")
        for c in conditions:
            lines.append(f"  - {c['condition']} (onset: {c['onset']}, status: {c['status']})")

    # Medications
    meds = extract_medications(pid)
    if meds:
        active_meds = [m for m in meds if m["status"] == "active"]
        if active_meds:
            lines.append("\nActive Medications:")
            # Deduplicate by name
            seen = set()
            for m in active_meds:
                if m["medication"] not in seen:
                    lines.append(f"  - {m['medication']}")
                    seen.add(m["medication"])

    # Key observations
    obs = extract_observations(pid, max_per_type=1)
    if obs:
        lines.append("\nRecent Observations:")
        for obs_name, values in list(obs.items())[:10]:
            v = values[0]
            lines.append(f"  - {obs_name}: {v['value']} ({v['date']})")

    return "\n".join(lines)


# Build summaries for all patients
patient_summaries = []
for p in patients:
    summary = build_patient_summary(p)
    patient_summaries.append({"id": p["id"], "name": p["name"], "summary": summary})

print(f"Built summaries for {len(patient_summaries)} patients.")

In [ ]:
# Display a sample patient summary
if patient_summaries:
    print("=" * 60)
    print("SAMPLE PATIENT SUMMARY")
    print("=" * 60)
    print(patient_summaries[0]["summary"])

---
## Step 5: Build RAG Knowledge Base

We create a small knowledge base of clinical guidelines. In a real system, this could contain thousands of documents — clinical practice guidelines, drug references, institutional protocols, etc.

We embed each document using `sentence-transformers` (`all-MiniLM-L6-v2`, ~80MB, CPU-only) and store the vectors in a numpy array for cosine similarity search. For larger knowledge bases (1000+ documents), consider a dedicated vector database like ChromaDB or FAISS.

In [ ]:
# Clinical guideline documents for the knowledge base
CLINICAL_GUIDELINES = [
    {
        "id": "ada-diabetes",
        "title": "ADA Standards of Care in Diabetes (2024)",
        "content": (
            "Type 2 Diabetes Management (ADA 2024): First-line therapy is metformin plus "
            "lifestyle modification. If HbA1c remains above 7% after 3 months, add a second "
            "agent based on comorbidities: SGLT2 inhibitor if heart failure or CKD is present, "
            "GLP-1 receptor agonist if atherosclerotic CVD or high CV risk. For patients with "
            "HbA1c > 10% or symptoms of hyperglycemia, consider early insulin therapy. "
            "Monitor HbA1c every 3 months until stable, then every 6 months. "
            "Blood pressure target: <130/80 mmHg. Statin therapy recommended for all diabetics "
            "age 40-75. Annual screening: dilated eye exam, urine albumin-to-creatinine ratio, "
            "foot exam, lipid panel."
        ),
    },
    {
        "id": "aha-hypertension",
        "title": "AHA/ACC Hypertension Guidelines",
        "content": (
            "Hypertension Management (AHA/ACC): Stage 1 hypertension (130-139/80-89 mmHg) "
            "with low CV risk: lifestyle modifications for 3-6 months. Stage 1 with high CV risk "
            "or Stage 2 (>=140/90): initiate pharmacotherapy. First-line agents: ACE inhibitors, "
            "ARBs, calcium channel blockers, or thiazide diuretics. For Black patients, preferred "
            "initial therapy is CCB or thiazide. If BP not at goal with one agent, add a second "
            "from a different class. Target BP <130/80 for most adults. Resistant hypertension "
            "(uncontrolled on 3 agents including a diuretic): add spironolactone. "
            "Lifestyle: DASH diet, sodium <1500mg/day, aerobic exercise 90-150 min/week, "
            "limit alcohol, weight loss if BMI >25."
        ),
    },
    {
        "id": "gina-asthma",
        "title": "GINA Asthma Management Guidelines",
        "content": (
            "Asthma Management (GINA 2024): Step 1 (mild intermittent): as-needed low-dose "
            "ICS-formoterol. Step 2 (mild persistent): daily low-dose ICS or as-needed "
            "ICS-formoterol. Step 3 (moderate): low-dose ICS-LABA maintenance. Step 4: "
            "medium-dose ICS-LABA. Step 5 (severe): high-dose ICS-LABA, consider add-on "
            "tiotropium, anti-IgE (omalizumab), anti-IL5 (mepolizumab), or anti-IL4R (dupilumab). "
            "IMPORTANT: GINA no longer recommends SABA-only treatment. All patients should "
            "receive ICS-containing therapy. Assess control using ACT score (>20 = well-controlled). "
            "Review inhaler technique and adherence before stepping up. "
            "For acute exacerbation: oral prednisolone 40-50mg for 5-7 days."
        ),
    },
    {
        "id": "gold-copd",
        "title": "GOLD COPD Management Guidelines",
        "content": (
            "COPD Management (GOLD 2024): Initial therapy based on symptoms and exacerbation "
            "history. Group A (low symptoms, low risk): SABA or SAMA as needed. Group B "
            "(more symptoms): LABA or LAMA. Group E (exacerbations): LABA+LAMA; if eosinophils "
            ">=300, consider LABA+LAMA+ICS triple therapy. Smoking cessation is the single most "
            "effective intervention. Pulmonary rehabilitation recommended for all symptomatic "
            "patients. Annual influenza and pneumococcal vaccination. "
            "For acute exacerbation: short-course systemic corticosteroids (prednisone 40mg x 5 days), "
            "antibiotics if increased sputum purulence, bronchodilators via nebulizer. "
            "Oxygen therapy if PaO2 <55 mmHg or SpO2 <88%."
        ),
    },
    {
        "id": "drug-interactions",
        "title": "Common Drug Interactions in Primary Care",
        "content": (
            "Key Drug Interactions: (1) ACE inhibitors + potassium-sparing diuretics: risk of "
            "hyperkalemia, monitor K+ levels. (2) Warfarin + NSAIDs: increased bleeding risk, "
            "avoid combination or use PPI prophylaxis. (3) Metformin + contrast dye: risk of "
            "lactic acidosis, hold metformin 48h before and after contrast. (4) SSRIs + triptans: "
            "serotonin syndrome risk, use with caution. (5) Statins + macrolide antibiotics: "
            "increased statin levels and myopathy risk, use azithromycin (safer) over clarithromycin. "
            "(6) ACE inhibitors + ARBs: dual RAAS blockade increases renal failure risk, avoid. "
            "(7) Fluoroquinolones + QT-prolonging drugs: arrhythmia risk. "
            "(8) Opioids + benzodiazepines: respiratory depression, FDA black box warning."
        ),
    },
    {
        "id": "uspstf-screening",
        "title": "USPSTF Preventive Screening Recommendations",
        "content": (
            "Preventive Screening (USPSTF): Colorectal cancer: screening ages 45-75, colonoscopy "
            "every 10 years or annual FIT. Breast cancer: mammography every 2 years ages 40-74. "
            "Cervical cancer: Pap smear every 3 years (21-29) or Pap+HPV co-test every 5 years "
            "(30-65). Lung cancer: annual low-dose CT for adults 50-80 with 20+ pack-year smoking "
            "history. Depression: screen all adults, PHQ-2 then PHQ-9. Diabetes: screen adults "
            "35-70 who are overweight/obese. Hypertension: screen all adults >= 18. "
            "Hepatitis C: screen all adults 18-79. HIV: screen all adults 15-65. "
            "Abdominal aortic aneurysm: one-time ultrasound for men 65-75 who have ever smoked."
        ),
    },
]

print(f"Knowledge base: {len(CLINICAL_GUIDELINES)} guideline documents")
for g in CLINICAL_GUIDELINES:
    print(f"  - {g['title']}")

In [ ]:
# Embed guidelines and store in a simple in-memory vector store
print("Loading embedding model (all-MiniLM-L6-v2, ~80MB)...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# Embed all guideline documents
guideline_texts = [g["content"] for g in CLINICAL_GUIDELINES]
guideline_embeddings = embedder.encode(guideline_texts)  # numpy array (N, 384)


def search_guidelines(query_text, n_results=3):
    """Retrieve the top-n most similar guidelines using cosine similarity."""
    query_emb = embedder.encode([query_text])  # (1, 384)
    # Cosine similarity = dot product of normalized vectors
    norms_q = np.linalg.norm(query_emb, axis=1, keepdims=True)
    norms_d = np.linalg.norm(guideline_embeddings, axis=1, keepdims=True)
    similarities = (query_emb @ guideline_embeddings.T) / (norms_q * norms_d.T)
    top_indices = np.argsort(similarities[0])[::-1][:n_results]
    return [
        {"index": int(i), "similarity": float(similarities[0][i]), **CLINICAL_GUIDELINES[i]}
        for i in top_indices
    ]


print(f"Embedded {len(guideline_texts)} guideline documents (shape: {guideline_embeddings.shape}).")

In [ ]:
# Test retrieval with a sample query
test_query = "patient with diabetes and high blood pressure, what medications?"

results = search_guidelines(test_query, n_results=3)

print(f"Query: '{test_query}'\n")
print("Top 3 retrieved guidelines:")
for i, r in enumerate(results, 1):
    print(f"  {i}. {r['title']} (similarity: {r['similarity']:.3f})")

---
## Step 6: Clinical Reasoning with RAG + Ollama

Now we connect everything: given a patient summary, we retrieve relevant guidelines from ChromaDB and ask Ollama to generate clinical reasoning grounded in those guidelines.

We use **LangChain** minimally — just for prompt templates and chain composition.

In [ ]:
# Connect to Ollama via LangChain
llm = OllamaLLM(
    model=OLLAMA_MODEL,
    base_url=OLLAMA_BASE_URL,
    temperature=0.3,  # Lower temperature for more factual responses
)

# Test connection
test_response = llm.invoke("Say 'hello' in one word.")
print(f"Ollama connection OK. Model: {OLLAMA_MODEL}")
print(f"Test response: {test_response.strip()}")

In [ ]:
# RAG prompt template
RAG_PROMPT = PromptTemplate.from_template(
"""You are a clinical decision support assistant. Using the patient record and
relevant clinical guidelines below, provide a brief clinical assessment.

IMPORTANT: Base your recommendations on the provided guidelines. Cite which
guideline supports each recommendation. If the guidelines do not cover a topic,
say so explicitly.

=== PATIENT RECORD ===
{patient_summary}

=== RELEVANT CLINICAL GUIDELINES ===
{guidelines}

=== CLINICAL QUESTION ===
{question}

Provide a structured response with:
1. Key clinical findings
2. Guideline-based recommendations (cite the guideline)
3. Potential drug interactions or safety concerns
4. Suggested follow-up actions
""")

NO_RAG_PROMPT = PromptTemplate.from_template(
"""You are a clinical decision support assistant. Based on the patient record
below, provide a brief clinical assessment.

=== PATIENT RECORD ===
{patient_summary}

=== CLINICAL QUESTION ===
{question}

Provide a structured response with:
1. Key clinical findings
2. Recommendations
3. Potential drug interactions or safety concerns
4. Suggested follow-up actions
""")


def clinical_rag_query(patient_summary, question, use_rag=True):
    """Generate a clinical assessment using optional RAG retrieval."""
    if use_rag:
        # Retrieve relevant guidelines
        query_text = f"{patient_summary}\n{question}"
        results = search_guidelines(query_text, n_results=3)

        guidelines_text = ""
        for r in results:
            guidelines_text += f"\n[{r['title']}]\n{r['content']}\n"

        chain = RAG_PROMPT | llm
        response = chain.invoke({
            "patient_summary": patient_summary,
            "guidelines": guidelines_text,
            "question": question,
        })
    else:
        chain = NO_RAG_PROMPT | llm
        response = chain.invoke({
            "patient_summary": patient_summary,
            "question": question,
        })

    return response

print("RAG query function defined.")

In [ ]:
# Key demo: side-by-side RAG vs no-RAG comparison
if patient_summaries:
    demo_patient = patient_summaries[0]
    question = "What are the key clinical concerns for this patient and what guideline-based actions should be taken?"

    print("=" * 70)
    print(f"PATIENT: {demo_patient['name']}")
    print("=" * 70)
    print(demo_patient["summary"])
    print()

    # Without RAG
    print("=" * 70)
    print("RESPONSE WITHOUT RAG (no guideline context)")
    print("=" * 70)
    no_rag_response = clinical_rag_query(demo_patient["summary"], question, use_rag=False)
    print(no_rag_response)

    print()

    # With RAG
    print("=" * 70)
    print("RESPONSE WITH RAG (guideline-grounded)")
    print("=" * 70)
    rag_response = clinical_rag_query(demo_patient["summary"], question, use_rag=True)
    print(rag_response)

In [ ]:
# Run RAG across multiple patients
question = "Summarize the top clinical priorities and recommended actions for this patient."

for i, ps in enumerate(patient_summaries[:3], 1):
    print("=" * 70)
    print(f"PATIENT {i}: {ps['name']}")
    print("=" * 70)
    response = clinical_rag_query(ps["summary"], question, use_rag=True)
    print(response)
    print()

---
## Step 7: In-Class Exercises & Discussion

### Pipeline Summary

| Component | Tool | Purpose |
|-----------|------|---------|
| Data generation | Synthea | Realistic synthetic FHIR patient records |
| Data storage | HAPI FHIR (Docker) | Standards-based clinical data server |
| Data access | `requests` (Python) | Raw REST API calls to FHIR endpoints |
| Embedding model | `all-MiniLM-L6-v2` | Convert text to vectors (~80MB, CPU) |
| Vector store | numpy (in-memory) | Cosine similarity search over guidelines |
| LLM | Ollama (`llama3.2`) | Local inference (~2GB, no API keys) |
| Orchestration | LangChain (minimal) | Prompt templates + chain composition |

### Exercise 1: Explore Different Patients

Pick a patient from the list and run the RAG query with a specific clinical question.
Try questions like:
- "Does this patient need any preventive screenings?"
- "Are there any potential drug interactions in this patient's medications?"
- "What lifestyle modifications should be recommended?"

In [ ]:
# Exercise 1: Try your own patient and question
# Choose a patient index (0 to len(patient_summaries)-1)
patient_idx = 0
my_question = "Does this patient need any preventive screenings based on their age and conditions?"

ps = patient_summaries[patient_idx]
print(f"Patient: {ps['name']}\n")
response = clinical_rag_query(ps["summary"], my_question, use_rag=True)
print(response)

### Exercise 2: Try Different Models or Prompts

1. Change `OLLAMA_MODEL` to a different model (e.g., `mistral`, `phi3`, `gemma2`) and re-run
2. Modify the `RAG_PROMPT` template to ask for different output formats
3. Adjust the `temperature` parameter (0.1 = more deterministic, 0.9 = more creative)

In [ ]:
# Exercise 2: Experiment with model settings
# Uncomment and modify to try different configurations

# Try a different model:
# llm_alt = OllamaLLM(model="mistral", base_url=OLLAMA_BASE_URL, temperature=0.3)

# Try a different temperature:
# llm_creative = OllamaLLM(model=OLLAMA_MODEL, base_url=OLLAMA_BASE_URL, temperature=0.9)

print("Modify this cell to experiment with different models and settings.")

### Discussion Questions

1. **RAG Quality**: How did the RAG response compare to the non-RAG response? What specific differences did you notice in terms of accuracy, specificity, and citation of sources?

2. **FHIR Completeness**: What information was available in the FHIR records vs what a clinician would need for real decision-making? What's missing?

3. **Safety & Regulation**: If you deployed a system like this in a clinical setting, what FDA/regulatory considerations would apply? How would you validate its outputs?

4. **Privacy**: We used synthetic data. What would change if we used real patient data? Consider HIPAA, de-identification, and the implications of sending patient data to an LLM.

5. **Embedding Model Choice**: We used `all-MiniLM-L6-v2` (general-purpose). Would a biomedical-specific embedding model (e.g., `PubMedBERT`) improve retrieval? How would you test this?

6. **Knowledge Base Scaling**: Our knowledge base has 6 documents. What challenges would arise with 10,000+ guidelines? Consider chunking strategies, retrieval precision, and context window limits.

---
### Cleanup

When you're done:
- Stop the HAPI FHIR Docker container: `docker stop <container_id>`
- Ollama can remain running or be stopped with `ollama stop`
- The vector store is in-memory and will be freed when the kernel restarts